# Train a tiny campus intent model from scratch
CPU runtime is enough; no GPU/API key/Drive mount/pretrained downloads. Run cells in order.
Upload all Claude JSONL batches together. Synthetic test accuracy is NOT proof of real student/phone quality.
This notebook contains the training code; no repository clone is needed. The exported classifier takes no navigation actions.


In [ ]:
from pathlib import Path
import json, os, sys
os.environ['OPENBLAS_NUM_THREADS']='2'
ROOT=Path('/content/ssn-campus-intents')
ROOT.mkdir(exist_ok=True)
FILES=json.loads('{"core.py": "\\"\\"\\"Dataset and feature contract shared by validation, training and browser tests.\\"\\"\\"\\nimport hashlib\\nimport json\\nimport re\\nimport unicodedata\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nFEATURES = {\\"bins\\": 2048, \\"min_n\\": 3, \\"max_n\\": 5, \\"max_chars\\": 200,\\n            \\"hash\\": \\"fnv1a32-utf8\\", \\"normalization\\": \\"nfkc-lower-punctuation-whitespace-v1\\"}\\nREJECT_LABELS = {\\"unknown\\", \\"out_of_scope\\"}\\n\\n\\ndef normalize(text):\\n    text = unicodedata.normalize(\\"NFKC\\", text).lower()\\n    return re.sub(r\\"\\\\s+\\", \\" \\", re.sub(r\\"[?!.,;:\'\\\\\\"]\\", \\" \\", text)).strip()[:FEATURES[\\"max_chars\\"]]\\n\\n\\ndef hash32(text):\\n    value = 2166136261\\n    for byte in text.encode(\\"utf-8\\"):\\n        value = ((value ^ byte) * 16777619) & 0xFFFFFFFF\\n    return value\\n\\n\\ndef feature_counts(text, config=FEATURES):\\n    text = normalize(text)\\n    counts = Counter()\\n    if not text:\\n        return counts\\n    padded = \\" \\" + text + \\" \\"\\n    for n in range(config[\\"min_n\\"], config[\\"max_n\\"] + 1):\\n        for start in range(len(padded) - n + 1):\\n            counts[hash32(\\"c:\\" + padded[start:start + n]) % config[\\"bins\\"]] += 1\\n    words = text.split()\\n    for word in words:\\n        counts[hash32(\\"w:\\" + word) % config[\\"bins\\"]] += 1\\n    for left, right in zip(words, words[1:]):\\n        counts[hash32(\\"b:\\" + left + \\" \\" + right) % config[\\"bins\\"]] += 1\\n    return counts\\n\\n\\ndef read_catalog(path):\\n    catalog = json.loads(Path(path).read_text(encoding=\\"utf-8\\"))\\n    return catalog, [item[\\"label\\"] for item in catalog[\\"labels\\"]]\\n\\n\\ndef family_key(text, aliases):\\n    # Finds identical templates with different campus names/numbers. It cannot\\n    # prove semantic independence; human-authored testing remains necessary.\\n    text = normalize(text)\\n    for alias in aliases:\\n        text = re.sub(r\\"(?<!\\\\w)\\" + re.escape(alias) + r\\"(?!\\\\w)\\", \\"<place>\\", text)\\n    return re.sub(r\\"\\\\d+\\", \\"<number>\\", text)\\n\\n\\ndef prototype_catalog(paths, catalog):\\n    \\"\\"\\"Explicit partial-data experiment; never used by full-dataset validation.\\"\\"\\"\\n    supplied = {json.loads(line)[\\"label\\"] for path in paths\\n                for line in Path(path).read_text(encoding=\\"utf-8-sig\\").splitlines() if line.strip()}\\n    missing = [item[\\"label\\"] for item in catalog[\\"labels\\"] if item[\\"label\\"] not in supplied]\\n    subset = {**catalog, \\"labels\\": [item for item in catalog[\\"labels\\"] if item[\\"label\\"] in supplied]}\\n    if len(subset[\\"labels\\"]) < 2:\\n        raise ValueError(\\"Prototype needs at least two known labels\\")\\n    return subset, missing\\n\\n\\ndef load_dataset(paths, catalog, allow_small=False):\\n    labels = {item[\\"label\\"] for item in catalog[\\"labels\\"]}\\n    aliases = sorted({normalize(alias) for place in catalog[\\"locations\\"]\\n                      for alias in [place[\\"name\\"], place[\\"id\\"].replace(\\"-\\", \\" \\"), *place[\\"aliases\\"]]\\n                      if len(normalize(alias)) > 2}, key=len, reverse=True)\\n    rows, identifiers, texts, groups, families = [], set(), {}, {}, {}\\n    for path in paths:\\n        for line_number, line in enumerate(Path(path).read_text(encoding=\\"utf-8-sig\\").splitlines(), 1):\\n            if not line.strip():\\n                continue\\n            try:\\n                row = json.loads(line)\\n            except json.JSONDecodeError as error:\\n                raise ValueError(f\\"{path}:{line_number}: invalid JSONL\\") from error\\n            if not isinstance(row, dict):\\n                raise ValueError(f\\"{path}:{line_number}: expected an object\\")\\n            for field in (\\"id\\", \\"text\\", \\"label\\", \\"group\\", \\"split\\", \\"language\\"):\\n                if not isinstance(row.get(field), str) or not row[field].strip():\\n                    raise ValueError(f\\"{path}:{line_number}: missing/non-string {field}\\")\\n            if len(row[\\"text\\"]) > FEATURES[\\"max_chars\\"] or not normalize(row[\\"text\\"]):\\n                raise ValueError(f\\"{row[\'id\']}: text must contain 1..200 characters\\")\\n            if row[\\"id\\"] in identifiers:\\n                raise ValueError(f\\"Duplicate id: {row[\'id\']}\\")\\n            if row[\\"label\\"] not in labels or row[\\"split\\"] not in (\\"train\\", \\"validation\\", \\"test\\"):\\n                raise ValueError(f\\"{row[\'id\']}: unknown label or split\\")\\n            if row[\\"language\\"] not in (\\"en\\", \\"tanglish\\"):\\n                raise ValueError(f\\"{row[\'id\']}: language must be en or tanglish\\")\\n            key = normalize(row[\\"text\\"])\\n            if key in texts:\\n                raise ValueError(f\\"Duplicate normalized text: {row[\'id\']} and {texts[key]}\\")\\n            if row[\\"group\\"] in groups and groups[row[\\"group\\"]] != row[\\"split\\"]:\\n                raise ValueError(f\\"Group leaks across splits: {row[\'group\']}\\")\\n            family = family_key(row[\\"text\\"], aliases)\\n            if family in families and families[family] != row[\\"split\\"]:\\n                raise ValueError(f\\"Location/number template leaks across splits: {row[\'id\']}\\")\\n            identifiers.add(row[\\"id\\"])\\n            texts[key], groups[row[\\"group\\"]], families[family] = row[\\"id\\"], row[\\"split\\"], row[\\"split\\"]\\n            rows.append(row)\\n    counts = Counter((row[\\"label\\"], row[\\"split\\"]) for row in rows)\\n    minimums = {\\"train\\": 2, \\"validation\\": 1, \\"test\\": 1} if allow_small else {\\"train\\": 60, \\"validation\\": 10, \\"test\\": 10}\\n    for label in sorted(labels):\\n        for split, minimum in minimums.items():\\n            if counts[label, split] < minimum:\\n                raise ValueError(f\\"{label}/{split}: {counts[label, split]} rows; need at least {minimum}\\")\\n    summary = {\\"rows\\": len(rows), \\"labels\\": len(labels),\\n               \\"splits\\": dict(Counter(row[\\"split\\"] for row in rows)),\\n               \\"languages\\": dict(Counter(row[\\"language\\"] for row in rows)),\\n               \\"groups\\": len(groups), \\"allow_small\\": allow_small,\\n               \\"sha256\\": hashlib.sha256(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode()).hexdigest()}\\n    return rows, summary\\n\\n\\ndef metrics(truth, predicted, labels, accepted):\\n    count = len(truth)\\n    per_class = {}\\n    for label in labels:\\n        tp = sum(t == p == label for t, p in zip(truth, predicted))\\n        fp = sum(t != label and p == label for t, p in zip(truth, predicted))\\n        fn = sum(t == label and p != label for t, p in zip(truth, predicted))\\n        per_class[label] = {\\"support\\": truth.count(label), \\"precision\\": tp / max(1, tp + fp),\\n                            \\"recall\\": tp / max(1, tp + fn), \\"f1\\": 2 * tp / max(1, 2 * tp + fp + fn)}\\n    total_accepted = sum(accepted)\\n    rejects = sum(t in REJECT_LABELS for t in truth)\\n    return {\\"samples\\": count, \\"accuracy\\": sum(t == p for t, p in zip(truth, predicted)) / max(1, count),\\n            \\"macro_f1\\": sum(item[\\"f1\\"] for item in per_class.values()) / len(labels),\\n            \\"coverage\\": total_accepted / max(1, count),\\n            \\"accepted_precision\\": sum(t == p and a for t, p, a in zip(truth, predicted, accepted)) / max(1, total_accepted),\\n            \\"out_of_scope_false_accept_rate\\": sum(t in REJECT_LABELS and a for t, a in zip(truth, accepted)) / max(1, rejects),\\n            \\"per_class\\": per_class}\\n", "train.py": "\\"\\"\\"CPU training from scratch. NumPy only; no pretrained weights or live APIs.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport os\\nimport shutil\\nimport time\\nimport zipfile\\nfrom pathlib import Path\\n\\nos.environ.setdefault(\\"OPENBLAS_NUM_THREADS\\", \\"2\\")\\nimport numpy as np  # noqa: E402\\n\\nfrom core import FEATURES, REJECT_LABELS, feature_counts, load_dataset, metrics, read_catalog, prototype_catalog  # noqa: E402\\n\\n\\ndef vectorize(rows):\\n    x = np.zeros((len(rows), FEATURES[\\"bins\\"]), dtype=np.float32)\\n    for index, row in enumerate(rows):\\n        for bucket, count in feature_counts(row[\\"text\\"]).items():\\n            x[index, bucket] = np.log1p(count)\\n    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)\\n\\n\\ndef probabilities(x, weights):\\n    w1, b1, w2, b2 = weights\\n    logits = np.maximum(0, x @ w1 + b1) @ w2 + b2\\n    exp = np.exp(logits - logits.max(axis=1, keepdims=True))\\n    return exp / exp.sum(axis=1, keepdims=True)\\n\\n\\ndef loss_and_gradients(x, y, weights):\\n    w1, b1, w2, b2 = weights\\n    z = x @ w1 + b1\\n    hidden = np.maximum(0, z)\\n    logits = hidden @ w2 + b2\\n    exp = np.exp(logits - logits.max(axis=1, keepdims=True))\\n    probs = exp / exp.sum(axis=1, keepdims=True)\\n    loss = -np.log(np.maximum(probs[np.arange(len(y)), y], 1e-12)).mean()\\n    delta = probs.copy()\\n    delta[np.arange(len(y)), y] -= 1\\n    delta /= len(y)\\n    dh = (delta @ w2.T) * (z > 0)\\n    return float(loss), [x.T @ dh, dh.sum(axis=0), hidden.T @ delta, delta.sum(axis=0)]\\n\\n\\ndef decisions(probs, labels, confidence, margin):\\n    order = np.argsort(-probs, axis=1, kind=\\"stable\\")\\n    top, runner = order[:, 0], order[:, 1]\\n    accepted = [(labels[t] not in REJECT_LABELS and float(probs[i, t]) >= confidence\\n                 and float(probs[i, t] - probs[i, r]) >= margin) for i, (t, r) in enumerate(zip(top, runner))]\\n    return [labels[i] for i in top], accepted\\n\\n\\ndef calibrate(probs, truth, labels):\\n    # Validation only. Test labels must never choose thresholds or weights.\\n    # No negative examples means rejection reliability was not measured.\\n    if not REJECT_LABELS.issubset(set(truth)):\\n        predicted, accepted = decisions(probs, labels, 1.01, 1.01)\\n        return {\\"enabled\\": False, \\"confidence\\": 1.01, \\"margin\\": 1.01}, metrics(truth, predicted, labels, accepted)\\n    best = None\\n    for confidence in (0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95):\\n        for margin in (0.05, 0.10, 0.15, 0.20):\\n            predicted, accepted = decisions(probs, labels, confidence, margin)\\n            score = metrics(truth, predicted, labels, accepted)\\n            if sum(accepted) >= 10 and score[\\"accepted_precision\\"] >= 0.97 and score[\\"out_of_scope_false_accept_rate\\"] <= 0.02:\\n                if best is None or score[\\"coverage\\"] > best[2][\\"coverage\\"]:\\n                    best = (confidence, margin, score)\\n    if best is None:\\n        predicted, accepted = decisions(probs, labels, 1.01, 1.01)\\n        return {\\"enabled\\": False, \\"confidence\\": 1.01, \\"margin\\": 1.01}, metrics(truth, predicted, labels, accepted)\\n    return {\\"enabled\\": True, \\"confidence\\": best[0], \\"margin\\": best[1]}, best[2]\\n\\n\\ndef train(args):\\n    root = Path(__file__).resolve().parent\\n    catalog, labels = read_catalog(root / \\"catalog.json\\")\\n    missing = []\\n    if getattr(args, \\"prototype\\", False):\\n        # Explicit subset experiment: retain the normal count/leakage checks.\\n        catalog, missing = prototype_catalog(args.files, catalog)\\n        labels = [item[\\"label\\"] for item in catalog[\\"labels\\"]]\\n    rows, summary = load_dataset(args.files, catalog, args.allow_small)\\n    summary.update(prototype=getattr(args, \\"prototype\\", False), missing_labels=missing)\\n    groups = {split: [row for row in rows if row[\\"split\\"] == split] for split in (\\"train\\", \\"validation\\", \\"test\\")}\\n    features = {split: vectorize(data) for split, data in groups.items()}\\n    targets = {split: np.array([labels.index(row[\\"label\\"]) for row in data]) for split, data in groups.items()}\\n    rng = np.random.default_rng(args.seed)\\n    weights = [(rng.standard_normal((FEATURES[\\"bins\\"], 64)) * np.sqrt(2 / FEATURES[\\"bins\\"])).astype(np.float32),\\n               np.zeros(64, dtype=np.float32), (rng.standard_normal((64, len(labels))) * np.sqrt(2 / 64)).astype(np.float32),\\n               np.zeros(len(labels), dtype=np.float32)]\\n    first, second = [np.zeros_like(w) for w in weights], [np.zeros_like(w) for w in weights]\\n    best, best_loss, stale, step = None, float(\\"inf\\"), 0, 0\\n    history, started = [], time.monotonic()\\n    for epoch in range(args.epochs):\\n        order = rng.permutation(len(groups[\\"train\\"]))\\n        for start in range(0, len(order), 64):\\n            batch = order[start:start + 64]\\n            _, grads = loss_and_gradients(features[\\"train\\"][batch], targets[\\"train\\"][batch], weights)\\n            step += 1\\n            for index, (weight, grad) in enumerate(zip(weights, grads)):\\n                first[index] = 0.9 * first[index] + 0.1 * grad\\n                second[index] = 0.999 * second[index] + 0.001 * grad * grad\\n                update = (first[index] / (1 - 0.9 ** step)) / (np.sqrt(second[index] / (1 - 0.999 ** step)) + 1e-8)\\n                weight -= 0.002 * (update + (0.0001 * weight if index in (0, 2) else 0))\\n        val = probabilities(features[\\"validation\\"], weights)\\n        val_loss = float(-np.log(np.maximum(val[np.arange(len(val)), targets[\\"validation\\"]], 1e-12)).mean())\\n        if not np.isfinite(val_loss):\\n            raise ValueError(\\"Training diverged; no model exported\\")\\n        history.append({\\"epoch\\": epoch + 1, \\"validation_loss\\": val_loss})\\n        print(f\\"Epoch {epoch + 1}: validation loss {val_loss:.4f}\\", flush=True)\\n        if val_loss < best_loss - 0.0001:\\n            best_loss, best, stale = val_loss, [w.copy() for w in weights], 0\\n        else:\\n            stale += 1\\n            if stale >= 7:\\n                break\\n    weights = best\\n    val_probs = probabilities(features[\\"validation\\"], weights)\\n    policy, validation = calibrate(val_probs, [r[\\"label\\"] for r in groups[\\"validation\\"]], labels)\\n    if getattr(args, \\"prototype\\", False):\\n        policy = {\\"enabled\\": False, \\"confidence\\": 1.01, \\"margin\\": 1.01}\\n        predicted, accepted = decisions(val_probs, labels, 1.01, 1.01)\\n        validation = metrics([r[\\"label\\"] for r in groups[\\"validation\\"]], predicted, labels, accepted)\\n    test_probs = probabilities(features[\\"test\\"], weights)\\n    predicted, accepted = decisions(test_probs, labels, policy[\\"confidence\\"], policy[\\"margin\\"])\\n    test = metrics([r[\\"label\\"] for r in groups[\\"test\\"]], predicted, labels, accepted)\\n    output = Path(args.output).resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    binary = b\\"\\".join(w.astype(\\"<f4\\").tobytes() for w in weights)\\n    if len(binary) > 1_000_000:\\n        raise ValueError(\\"Model exceeds the 1 MB weight budget\\")\\n    (output / \\"weights.bin\\").write_bytes(binary)\\n    sha = hashlib.sha256(binary).hexdigest()\\n    model = {\\"schema\\": 1, \\"model_id\\": \\"campus-\\" + sha[:12], \\"features\\": FEATURES, \\"hidden\\": 64,\\n             \\"labels\\": labels, \\"weights\\": {\\"file\\": \\"weights.bin\\", \\"bytes\\": len(binary), \\"sha256\\": sha},\\n             \\"decision\\": policy, \\"training\\": {\\"seed\\": args.seed, \\"dataset_sha256\\": summary[\\"sha256\\"], \\"from_scratch\\": True,\\n                                             \\"prototype\\": summary[\\"prototype\\"], \\"missing_labels\\": missing},\\n             \\"approved_for_production\\": False}\\n    report = {\\"dataset\\": summary, \\"validation\\": validation, \\"test\\": test, \\"history\\": history,\\n              \\"seconds\\": time.monotonic() - started, \\"numpy_version\\": np.__version__, \\"weight_bytes\\": len(binary),\\n              \\"decision\\": policy, \\"production_ready\\": False,\\n              \\"remaining\\": [\\"Review synthetic labels and obtain independently written student questions.\\",\\n                            \\"Benchmark on actual Android/iOS phones alongside map navigation.\\",\\n                            \\"Do not execute actions from a label without entity/context validation.\\"]}\\n    for name, value in ((\\"model.json\\", model), (\\"report.json\\", report), (\\"catalog.json\\", catalog)):\\n        (output / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding=\\"utf-8\\")\\n    cases = []\\n    for text in [r[\\"text\\"] for r in groups[\\"test\\"][:20]] + [\\"\\", \\"ECE 302\\", \\"food enga iruku?\\", \\"\\uff24\\uff4f\\uff4e\\u2019\\uff54 stop navigation\\", \\"\\u0ba4\\u0bae\\u0bbf\\u0bb4\\u0bcd\\"]:\\n        probs = probabilities(vectorize([{\\"text\\": text}]), weights)[0]\\n        cases.append({\\"text\\": text, \\"probabilities\\": probs.tolist(), \\"counts\\": dict(feature_counts(text))})\\n    (output / \\"conformance.json\\").write_text(json.dumps(cases, ensure_ascii=False), encoding=\\"utf-8\\")\\n    for source in (root / \\"phone-test\\").iterdir():\\n        shutil.copy2(source, output / source.name)\\n    archive = output.parent / \\"model_bundle.zip\\"\\n    with zipfile.ZipFile(archive, \\"w\\", zipfile.ZIP_DEFLATED) as bundle:\\n        for file in output.iterdir():\\n            if file.is_file():\\n                bundle.write(file, file.name)\\n    print(json.dumps({\\"test_accuracy\\": test[\\"accuracy\\"], \\"macro_f1\\": test[\\"macro_f1\\"],\\n                      \\"accepted_precision\\": test[\\"accepted_precision\\"], \\"coverage\\": test[\\"coverage\\"],\\n                      \\"model_enabled\\": policy[\\"enabled\\"], \\"weight_bytes\\": len(binary), \\"bundle\\": str(archive)}, indent=2))\\n    return model, report\\n\\n\\nif __name__ == \\"__main__\\":\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"files\\", nargs=\\"+\\")\\n    parser.add_argument(\\"--output\\", default=\\"trained_model\\")\\n    parser.add_argument(\\"--epochs\\", type=int, default=40)\\n    parser.add_argument(\\"--seed\\", type=int, default=42)\\n    parser.add_argument(\\"--allow-small\\", action=\\"store_true\\", help=\\"Fixture smoke tests only\\")\\n    parser.add_argument(\\"--prototype\\", action=\\"store_true\\", help=\\"Train supplied labels only; disable acceptance; never a production model\\")\\n    arguments = parser.parse_args()\\n    if not 1 <= arguments.epochs <= 200:\\n        parser.error(\\"--epochs must be between 1 and 200\\")\\n    train(arguments)\\n", "validate_dataset.py": "\\"\\"\\"Usage: python validate_dataset.py batch*.jsonl [--allow-small].\\"\\"\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nfrom core import load_dataset, read_catalog, prototype_catalog\\n\\nparser = argparse.ArgumentParser(description=__doc__)\\nparser.add_argument(\\"files\\", nargs=\\"+\\")\\nparser.add_argument(\\"--catalog\\", default=str(Path(__file__).with_name(\\"catalog.json\\")))\\nparser.add_argument(\\"--allow-small\\", action=\\"store_true\\", help=\\"Fixtures only; not a production dataset\\")\\nparser.add_argument(\\"--prototype\\", action=\\"store_true\\", help=\\"Validate supplied labels only; report missing labels; not production-ready\\")\\nargs = parser.parse_args()\\ncatalog, _ = read_catalog(args.catalog)\\nmissing = []\\nif args.prototype:\\n    catalog, missing = prototype_catalog(args.files, catalog)\\n_, summary = load_dataset(args.files, catalog, args.allow_small)\\nsummary.update(prototype=args.prototype, missing_labels=missing)\\nprint(json.dumps(summary, indent=2))\\nprint(\\"PASS: labels, IDs, exact duplicates, group/template leakage and split coverage\\")\\n", "catalog.json": "{\\n  \\"schema\\": 1,\\n  \\"task\\": \\"intent classification only\\",\\n  \\"labels\\": [\\n    {\\n      \\"label\\": \\"greeting\\",\\n      \\"description\\": \\"A greeting or thanks without another substantive request.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"greeting\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"building_finder\\",\\n      \\"description\\": \\"Find a named campus building or get directions to it. Resolve the location separately.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"building_finder\\",\\n        \\"requires_entities\\": [\\n          \\"location\\"\\n        ]\\n      }\\n    },\\n    {\\n      \\"label\\": \\"building_info\\",\\n      \\"description\\": \\"Ask for a description of a named campus building, rather than its route.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"building_finder\\",\\n        \\"info_mode\\": true,\\n        \\"requires_entities\\": [\\n          \\"location\\"\\n        ]\\n      }\\n    },\\n    {\\n      \\"label\\": \\"department_finder\\",\\n      \\"description\\": \\"Explicitly locate a department/dept. Bare CSE or ECE without department is building_finder.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"department_finder\\",\\n        \\"requires_entities\\": [\\n          \\"location\\"\\n        ]\\n      }\\n    },\\n    {\\n      \\"label\\": \\"classroom_finder\\",\\n      \\"description\\": \\"Find a room/lab using a department and room code, such as ECE 302 or CSE Lab 4. Do not label exact floor assertions.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"classroom_finder\\",\\n        \\"requires_entities\\": [\\n          \\"department\\",\\n          \\"room\\"\\n        ]\\n      }\\n    },\\n    {\\n      \\"label\\": \\"venue_menu\\",\\n      \\"description\\": \\"Ask what a food court serves or request its uploaded daily menu; not merely where to eat.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"venue_menu\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_now\\",\\n      \\"description\\": \\"Events currently happening, with an explicit now/currently meaning.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_now\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_today\\",\\n      \\"description\\": \\"Today\'s schedule, including later today/this afternoon; not an arbitrary date query.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_today\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_upcoming\\",\\n      \\"description\\": \\"Future upcoming events without the next-30-minutes constraint.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_upcoming\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_list\\",\\n      \\"description\\": \\"General list of fest events without a time or proximity filter.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_list\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_near_me\\",\\n      \\"description\\": \\"Events nearest to the user\'s position, without another incompatible filter.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_near_me\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"event_upcoming_30\\",\\n      \\"description\\": \\"Events starting within 30 minutes / starting soon.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"event_upcoming_30\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"nearby_search\\",\\n      \\"description\\": \\"General places around the user; a specific facility request uses need.* instead.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"nearby_search\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"distance_query\\",\\n      \\"description\\": \\"Distance/ETA between two explicitly named campus places.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"distance_query\\",\\n        \\"requires_entities\\": [\\n          \\"from\\",\\n          \\"to\\"\\n        ]\\n      }\\n    },\\n    {\\n      \\"label\\": \\"current_location\\",\\n      \\"description\\": \\"Ask for this device\'s current position.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"current_location\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.dining\\",\\n      \\"description\\": \\"Find somewhere to eat or drink, not its menu or nutritional facts.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"dining\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.water_station\\",\\n      \\"description\\": \\"Find drinking water/refill point.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"water_station\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.restroom\\",\\n      \\"description\\": \\"Find a toilet/restroom/washroom.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"restroom\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.parking\\",\\n      \\"description\\": \\"Find vehicle parking.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"parking\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.medical\\",\\n      \\"description\\": \\"Locate the medical center or first-aid location; never diagnose or prescribe.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"medical\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.accessibility\\",\\n      \\"description\\": \\"Find buildings marked wheelchair accessible; not a guarantee of a step-free route.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"accessibility\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"need.placement\\",\\n      \\"description\\": \\"Locate the placement office / Career Development Cell (CDC).\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"need\\",\\n        \\"need_type\\": \\"placement\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_navigate\\",\\n      \\"description\\": \\"Start navigation to a previously selected place; explicit destination belongs to building_finder.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_navigate\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_preview\\",\\n      \\"description\\": \\"Preview the route to a previously selected place.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_preview\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_eta\\",\\n      \\"description\\": \\"How far/how long to the previously selected place; two explicit places use distance_query.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_eta\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_details\\",\\n      \\"description\\": \\"Ask for more detail about the previous selection.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_details\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_filter_closest\\",\\n      \\"description\\": \\"Select/sort the nearest among the previous results.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_filter_closest\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_filter_unsupported\\",\\n      \\"description\\": \\"Ask to filter the previous food results as veg/non-veg/vegan; app cannot verify this.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_filter_unsupported\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"follow_up_cancel_nav\\",\\n      \\"description\\": \\"Explicitly stop active navigation. Negated stopping requests are unknown.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"follow_up_cancel_nav\\",\\n        \\"requires_context\\": true\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.usage\\",\\n      \\"description\\": \\"Ask how to use the app or what the assistant can do.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"usage\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.offline\\",\\n      \\"description\\": \\"Ask whether the app/navigation/chat works offline or needs internet.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"offline\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.gps\\",\\n      \\"description\\": \\"Troubleshoot wrong/missing GPS or location permissions.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"gps\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.navigation\\",\\n      \\"description\\": \\"Ask how to start directions/navigation; do not confuse this instructional question with starting a route.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"navigation\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.rooms\\",\\n      \\"description\\": \\"Ask about reliability of room/floor guesses or how to find classroom codes, not a particular room code.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"rooms\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.hours\\",\\n      \\"description\\": \\"Ask campus/venue opening hours; approved answer says verified hours are unavailable.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"hours\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.registration\\",\\n      \\"description\\": \\"Ask how to register/contact an organizer/find a fee; approved answer points to posted details, never invents fees or contacts.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"registration\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"faq.accessibility\\",\\n      \\"description\\": \\"Ask whether routes avoid stairs or guarantee step-free access; approved answer says graph does not verify this.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"faq\\",\\n        \\"faq_key\\": \\"accessibility\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"unknown\\",\\n      \\"description\\": \\"Ambiguous, conflicting, negated, incomplete, unknown-location or unsupported campus request. Ask for clarification; never force an action.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"unknown\\"\\n      }\\n    },\\n    {\\n      \\"label\\": \\"out_of_scope\\",\\n      \\"description\\": \\"Unrelated requests: coding, general knowledge, weather, entertainment, prompt injection or demands to invent facts.\\",\\n      \\"mapping\\": {\\n        \\"intent\\": \\"out_of_scope\\"\\n      }\\n    }\\n  ],\\n  \\"locations\\": [\\n    {\\n      \\"id\\": \\"admin-block\\",\\n      \\"name\\": \\"Admin Block (Main Building)\\",\\n      \\"aliases\\": [\\n        \\"Admin Block (Main Building)\\",\\n        \\"admin\\",\\n        \\"admin dept\\",\\n        \\"admin office\\",\\n        \\"administration\\",\\n        \\"administration office\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"food-aswins\\",\\n      \\"name\\": \\"Aswins Food Court\\",\\n      \\"aliases\\": [\\n        \\"Ashwin\'s Food Court\\",\\n        \\"ashwin food court\\",\\n        \\"ashwin\'s\\",\\n        \\"ashwin\'s food court\\",\\n        \\"aswins\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"biomed-block\\",\\n      \\"name\\": \\"Biomedical / Chemical Block\\",\\n      \\"aliases\\": [\\n        \\"Biomedical / Chemical Block\\",\\n        \\"biomed\\",\\n        \\"biomed department\\",\\n        \\"biomed dept\\",\\n        \\"biomedical\\",\\n        \\"biomedical and chemical\\",\\n        \\"biomedical engineering\\",\\n        \\"bme\\",\\n        \\"chemical\\",\\n        \\"chemical engineering\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"boys-hostel-gate\\",\\n      \\"name\\": \\"Boys Hostel - Gate\\",\\n      \\"aliases\\": [\\n        \\"Boys Hostel - Gate\\",\\n        \\"boys hostel\\",\\n        \\"boys hostel gate\\",\\n        \\"men\'s hostel\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"boys-hostel-office\\",\\n      \\"name\\": \\"Boys Hostel - Office\\",\\n      \\"aliases\\": [\\n        \\"Boys Hostel - Office\\",\\n        \\"boys hostel office\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"cdc-block\\",\\n      \\"name\\": \\"CDC - Career Development Cell\\",\\n      \\"aliases\\": [\\n        \\"CDC - Career Development Cell\\",\\n        \\"career cell\\",\\n        \\"career development\\",\\n        \\"career development cell\\",\\n        \\"cdc\\",\\n        \\"placement\\",\\n        \\"placement cell\\",\\n        \\"placement office\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"central-library\\",\\n      \\"name\\": \\"Central Library (CS Block)\\",\\n      \\"aliases\\": [\\n        \\"Central Library (CS Block)\\",\\n        \\"central library\\",\\n        \\"lib\\",\\n        \\"library\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"civil-block\\",\\n      \\"name\\": \\"Civil Block\\",\\n      \\"aliases\\": [\\n        \\"Civil Block\\",\\n        \\"civil\\",\\n        \\"civil department\\",\\n        \\"civil dept\\",\\n        \\"civil engineering\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"cse-block\\",\\n      \\"name\\": \\"CSE Block\\",\\n      \\"aliases\\": [\\n        \\"CSE Block\\",\\n        \\"comp sci\\",\\n        \\"computer science\\",\\n        \\"computer science engineering\\",\\n        \\"cs\\",\\n        \\"cse\\",\\n        \\"cse department\\",\\n        \\"cse dept\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"ece-block\\",\\n      \\"name\\": \\"ECE Block\\",\\n      \\"aliases\\": [\\n        \\"ECE Block\\",\\n        \\"ece\\",\\n        \\"ece department\\",\\n        \\"ece dept\\",\\n        \\"electronics and communication\\",\\n        \\"electronics communication\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"eee-block\\",\\n      \\"name\\": \\"EEE Block\\",\\n      \\"aliases\\": [\\n        \\"EEE Block\\",\\n        \\"eee\\",\\n        \\"eee block\\",\\n        \\"eee department\\",\\n        \\"eee dept\\",\\n        \\"electrical\\",\\n        \\"electrical and electronics\\",\\n        \\"electrical engineering\\",\\n        \\"electronics and electrical\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"girls-hostel\\",\\n      \\"name\\": \\"Girls Hostel (Ladies Hostel)\\",\\n      \\"aliases\\": [\\n        \\"Girls Hostel (Ladies Hostel)\\",\\n        \\"girls hostel\\",\\n        \\"ladies hostel\\",\\n        \\"women\'s hostel\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"it-block\\",\\n      \\"name\\": \\"IT Block\\",\\n      \\"aliases\\": [\\n        \\"IT Block\\",\\n        \\"information technology\\",\\n        \\"it department\\",\\n        \\"it dept\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"main-canteen\\",\\n      \\"name\\": \\"Main Canteen\\",\\n      \\"aliases\\": [\\n        \\"Main Canteen\\",\\n        \\"canteen\\",\\n        \\"main cafeteria\\",\\n        \\"main canteen\\",\\n        \\"main mess\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"main-gate\\",\\n      \\"name\\": \\"Main Gate / Entrance\\",\\n      \\"aliases\\": [\\n        \\"Main Gate / Entrance\\",\\n        \\"entrance\\",\\n        \\"entry\\",\\n        \\"gate\\",\\n        \\"main gate\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"mech-block\\",\\n      \\"name\\": \\"Mechanical Block\\",\\n      \\"aliases\\": [\\n        \\"Mechanical Block\\",\\n        \\"mech\\",\\n        \\"mech department\\",\\n        \\"mech dept\\",\\n        \\"mechanical\\",\\n        \\"mechanical engineering\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"medical-center\\",\\n      \\"name\\": \\"Medical Center (Dispensary)\\",\\n      \\"aliases\\": [\\n        \\"Medical Center (Dispensary)\\",\\n        \\"clinic\\",\\n        \\"hospital\\",\\n        \\"infirmary\\",\\n        \\"medical center\\",\\n        \\"medical centre\\",\\n        \\"sick bay\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"food-metro\\",\\n      \\"name\\": \\"Metro Cafe\\",\\n      \\"aliases\\": [\\n        \\"Metro Cafe\\",\\n        \\"metro\\",\\n        \\"metro cafe\\",\\n        \\"metro coffee\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"mini-hall-1\\",\\n      \\"name\\": \\"Mini Auditorium\\",\\n      \\"aliases\\": [\\n        \\"Mini Auditorium\\",\\n        \\"mini hall\\",\\n        \\"mini hall 1\\",\\n        \\"minihall\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"open-air-theatre\\",\\n      \\"name\\": \\"Open Air Theatre\\",\\n      \\"aliases\\": [\\n        \\"Open Air Theatre\\",\\n        \\"oat\\",\\n        \\"open air theater\\",\\n        \\"open air theatre\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"food-pr\\",\\n      \\"name\\": \\"PR Food Court\\",\\n      \\"aliases\\": [\\n        \\"PR Food Court\\",\\n        \\"pr\\",\\n        \\"pr canteen\\",\\n        \\"pr food court\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"food-rishabhs\\",\\n      \\"name\\": \\"Rishabh\'s Food Court\\",\\n      \\"aliases\\": [\\n        \\"Rishabh\'s Food Court\\",\\n        \\"rishabh food court\\",\\n        \\"rishabh\'s\\",\\n        \\"rishabh\'s food court\\",\\n        \\"rishabhs\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"snu-academic\\",\\n      \\"name\\": \\"Shiv Nadar University (SNU) Academic Block\\",\\n      \\"aliases\\": [\\n        \\"Shiv Nadar University (SNU) Academic Block\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"food-snowcube\\",\\n      \\"name\\": \\"Snow Cube\\",\\n      \\"aliases\\": [\\n        \\"Snow Cube\\",\\n        \\"snow cube\\",\\n        \\"snow cube cafe\\",\\n        \\"snowcube\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"sports-complex-main\\",\\n      \\"name\\": \\"Sports Complex\\",\\n      \\"aliases\\": [\\n        \\"Sports Complex\\",\\n        \\"main sports complex\\",\\n        \\"sports ground\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"sports-complex\\",\\n      \\"name\\": \\"Sports Complex & Ground\\",\\n      \\"aliases\\": [\\n        \\"Sports Complex & Ground\\",\\n        \\"gym\\",\\n        \\"sports\\",\\n        \\"sports complex\\",\\n        \\"stadium\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"ssn-cricket-ground\\",\\n      \\"name\\": \\"SSN Cricket Ground\\",\\n      \\"aliases\\": [\\n        \\"SSN Cricket Ground\\",\\n        \\"cricket field\\",\\n        \\"cricket ground\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"ssn-football-ground\\",\\n      \\"name\\": \\"SSN Football Ground\\",\\n      \\"aliases\\": [\\n        \\"SSN Football Ground\\",\\n        \\"football field\\",\\n        \\"football ground\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"ssn-fountain\\",\\n      \\"name\\": \\"SSN Fountain\\",\\n      \\"aliases\\": [\\n        \\"SSN Fountain\\",\\n        \\"fountain\\",\\n        \\"ssn fountain\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"clock-tower\\",\\n      \\"name\\": \\"SSN Clock Tower\\",\\n      \\"aliases\\": [\\n        \\"SSN Fountain / Clock Tower\\",\\n        \\"clock tower\\",\\n        \\"clocktower\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"tcs-auditorium\\",\\n      \\"name\\": \\"Main Auditorium\\",\\n      \\"aliases\\": [\\n        \\"TCS Auditorium (Main)\\",\\n        \\"auditorium\\",\\n        \\"main auditorium\\",\\n        \\"tcs\\",\\n        \\"tcs auditorium\\"\\n      ]\\n    },\\n    {\\n      \\"id\\": \\"parking\\",\\n      \\"name\\": \\"Visitor Parking\\",\\n      \\"aliases\\": [\\n        \\"Visitor Parking\\",\\n        \\"bike parking\\",\\n        \\"car park\\",\\n        \\"parking\\",\\n        \\"parking lot\\"\\n      ]\\n    }\\n  ],\\n  \\"notes\\": [\\n    \\"A mapping is not an executable action; resolve entities and validate context separately.\\",\\n    \\"No coordinates, changing events, credentials, phone numbers or campus facts are training targets.\\"\\n  ]\\n}", "phone-test/index.html": "<!doctype html>\\n<html lang=\\"en\\"><head><meta charset=\\"utf-8\\"><meta name=\\"viewport\\" content=\\"width=device-width,initial-scale=1\\">\\n<meta name=\\"robots\\" content=\\"noindex\\"><title>Campus model phone test</title>\\n<style>\\n:root{font:16px system-ui;color-scheme:light dark}body{max-width:640px;margin:auto;padding:20px;line-height:1.5}\\nbutton,input,textarea{font:inherit;box-sizing:border-box}button{min-height:44px;padding:8px 14px;margin:6px 4px 6px 0;cursor:pointer}\\ntextarea{width:100%;min-height:96px;padding:10px}input{max-width:100%}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:light-dark(#f1f4f8,#222);padding:12px;border-radius:8px}\\nfieldset{margin:16px 0;padding:12px}label{display:block;margin:10px 0}h1{font-size:1.4rem}\\n</style></head><body>\\n<h1>Campus model phone test</h1>\\n<p>Experimental intent classification only. This page does not send chat messages to a server, answer campus facts, or start navigation.</p>\\n<button id=\\"packaged\\">Load packaged model</button>\\n<fieldset><legend>Or choose the two Colab output files</legend>\\n<label>model.json <input id=\\"manifest\\" type=\\"file\\" accept=\\".json\\"></label>\\n<label>weights.bin <input id=\\"weights\\" type=\\"file\\" accept=\\".bin\\"></label>\\n<button id=\\"local\\">Load selected files</button></fieldset>\\n<p id=\\"status\\" role=\\"status\\">No model loaded.</p>\\n<label for=\\"question\\">Campus question (English or Tanglish)</label>\\n<textarea id=\\"question\\" maxlength=\\"200\\" placeholder=\\"Where can I get something to eat?\\"></textarea>\\n<button id=\\"predict\\" disabled>Classify question</button><button id=\\"benchmark\\" disabled>Benchmark 100 predictions</button>\\n<pre id=\\"result\\" aria-live=\\"polite\\">Results appear here.</pre>\\n<p>Record your phone, browser, loading time, p50/p95, and whether predictions match your intended question. High confidence alone does not establish correctness.</p>\\n<button id=\\"download\\">Download test log</button>\\n<script type=\\"module\\">\\nconst $ = id => document.getElementById(id), log = [], worker = new Worker(\'./worker.mjs\', {type:\'module\'})\\nlet loaded = false, busy = false\\nfunction controls() { for (const id of [\'packaged\',\'local\',\'predict\',\'benchmark\']) $(id).disabled = busy || (!loaded && [\'predict\',\'benchmark\'].includes(id)) }\\nfunction fail(error) { busy = false; $(\'status\').textContent = error.message || String(error); controls() }\\nfunction send(manifest, buffer) { loaded = false; busy = true; controls(); $(\'status\').textContent = \'Verifying and loading model\\u2026\'; worker.postMessage({type:\'load\', manifest, buffer}, [buffer]) }\\nworker.onmessage = ({data}) => {\\n  busy = false\\n  if (data.type === \'loaded\') { loaded = true; $(\'status\').textContent = `${data.id}: ${(data.bytes/1024).toFixed(0)} KB weights, loaded in ${data.ms.toFixed(1)} ms. ${data.enabled ? \'Confidence policy enabled.\' : \'Policy rejected validation quality; predictions are diagnostic only.\'}` }\\n  else if (data.type === \'error\') { fail(data.message); return }\\n  else $(\'result\').textContent = JSON.stringify(data, null, 2)\\n  log.push({time:new Date().toISOString(), ...data}); controls()\\n}\\nworker.onerror = event => fail(`Worker failed: ${event.message}. Serve this folder over HTTPS or localhost.`)\\n$(\'packaged\').onclick = async () => {\\n  busy = true; controls()\\n  try { const [a,b] = await Promise.all([fetch(\'./model.json\'),fetch(\'./weights.bin\')]); if (!a.ok || !b.ok) throw Error(\'Model files missing beside this page; choose files instead.\'); send(await a.json(),await b.arrayBuffer()) } catch(e) { fail(e) }\\n}\\n$(\'local\').onclick = async () => {\\n  try { const a=$(\'manifest\').files[0], b=$(\'weights\').files[0]; if (!a || !b) throw Error(\'Select model.json and weights.bin.\'); if (a.size > 100000 || b.size > 1000000) throw Error(\'Model exceeds size budget.\'); send(JSON.parse(await a.text()),await b.arrayBuffer()) } catch(e) { fail(e) }\\n}\\nfor (const type of [\'predict\',\'benchmark\']) $(type).onclick = () => { busy=true; controls(); log.push({question:$(\'question\').value}); worker.postMessage({type,text:$(\'question\').value}) }\\n$(\'download\').onclick = () => { const url=URL.createObjectURL(new Blob([JSON.stringify({browser:navigator.userAgent,log},null,2)],{type:\'application/json\'})); const a=document.createElement(\'a\'); a.href=url; a.download=\'campus-phone-test.json\'; a.click(); setTimeout(()=>URL.revokeObjectURL(url),1000) }\\n</script></body></html>\\n", "phone-test/runtime.mjs": "// Pure CPU inference. No imports, external APIs, or navigation actions.\\nexport function normalize(text) {\\n  return Array.from(String(text).normalize(\'NFKC\').toLowerCase()\\n    .replace(/[?!.,;:\'\\"]/g, \' \').replace(/\\\\s+/g, \' \').trim()).slice(0, 200).join(\'\')\\n}\\nconst encoder = new TextEncoder()\\nexport function hash32(text) {\\n  let hash = 2166136261\\n  for (const byte of encoder.encode(text)) hash = Math.imul(hash ^ byte, 16777619) >>> 0\\n  return hash\\n}\\nexport function featureCounts(text) {\\n  text = normalize(text)\\n  const counts = new Map()\\n  if (!text) return counts\\n  const add = part => { const index = hash32(part) % 2048; counts.set(index, (counts.get(index) || 0) + 1) }\\n  const chars = Array.from(` ${text} `)\\n  for (let n = 3; n <= 5; n++) for (let i = 0; i <= chars.length - n; i++) add(`c:${chars.slice(i, i + n).join(\'\')}`)\\n  const words = text.split(\' \')\\n  for (const word of words) add(`w:${word}`)\\n  for (let i = 0; i < words.length - 1; i++) add(`b:${words[i]} ${words[i + 1]}`)\\n  return counts\\n}\\nfunction features(text) {\\n  const x = new Float32Array(2048)\\n  let squared = 0\\n  for (const [index, count] of featureCounts(text)) { x[index] = Math.log1p(count); squared += x[index] ** 2 }\\n  const norm = Math.sqrt(squared) || 1\\n  for (let i = 0; i < x.length; i++) x[i] /= norm\\n  return x\\n}\\nexport async function loadModel(manifest, buffer) {\\n  if (manifest.schema !== 1 || manifest.hidden !== 64 || manifest.features?.bins !== 2048\\n    || manifest.features?.min_n !== 3 || manifest.features?.max_n !== 5 || manifest.features?.max_chars !== 200\\n    || manifest.features?.hash !== \'fnv1a32-utf8\'\\n    || manifest.features?.normalization !== \'nfkc-lower-punctuation-whitespace-v1\') throw Error(\'Unsupported model contract\')\\n  const labels = manifest.labels\\n  if (!Array.isArray(labels) || labels.length < 2 || labels.length > 128\\n    || !labels.every(l => typeof l === \'string\') || new Set(labels).size !== labels.length) throw Error(\'Invalid labels\')\\n  const expected = (2048 * 64 + 64 + 64 * labels.length + labels.length) * 4\\n  if (buffer.byteLength !== expected || buffer.byteLength > 1_000_000 || manifest.weights?.bytes !== expected) throw Error(\'Invalid weight size\')\\n  if (!globalThis.crypto?.subtle) throw Error(\'Serve this test over HTTPS or localhost to verify model integrity\')\\n  const digest = await crypto.subtle.digest(\'SHA-256\', buffer)\\n  const sha = [...new Uint8Array(digest)].map(b => b.toString(16).padStart(2, \'0\')).join(\'\')\\n  if (sha !== manifest.weights.sha256) throw Error(\'Weight SHA-256 mismatch\')\\n  const view = new DataView(buffer), values = new Float32Array(expected / 4)\\n  for (let i = 0; i < values.length; i++) {\\n    values[i] = view.getFloat32(i * 4, true)\\n    if (!Number.isFinite(values[i])) throw Error(\'Non-finite weights\')\\n  }\\n  const policy = manifest.decision\\n  if (!policy || typeof policy.enabled !== \'boolean\' || !Number.isFinite(policy.confidence)\\n    || !Number.isFinite(policy.margin) || policy.confidence < 0 || policy.confidence > 1.01\\n    || policy.margin < 0 || policy.margin > 1.01) throw Error(\'Invalid decision policy\')\\n  return { manifest, values }\\n}\\nexport function predict(model, text) {\\n  const { values, manifest } = model, labels = manifest.labels, count = labels.length\\n  const x = features(text), hidden = new Float64Array(64), logits = new Float64Array(count)\\n  const bias1 = 2048 * 64, weight2 = bias1 + 64, bias2 = weight2 + 64 * count\\n  for (let h = 0; h < 64; h++) {\\n    let value = values[bias1 + h]\\n    for (let i = 0; i < x.length; i++) if (x[i]) value += x[i] * values[i * 64 + h]\\n    hidden[h] = Math.max(0, value)\\n  }\\n  for (let c = 0; c < count; c++) {\\n    let value = values[bias2 + c]\\n    for (let h = 0; h < 64; h++) value += hidden[h] * values[weight2 + h * count + c]\\n    logits[c] = value\\n  }\\n  const max = Math.max(...logits), probabilities = [...logits].map(v => Math.exp(v - max))\\n  const total = probabilities.reduce((a, b) => a + b, 0)\\n  for (let i = 0; i < count; i++) probabilities[i] /= total\\n  const ranked = labels.map((label, i) => ({ label, score: probabilities[i] })).sort((a, b) => b.score - a.score)\\n  const confidence = ranked[0].score, margin = confidence - ranked[1].score\\n  return { label: ranked[0].label, confidence, margin,\\n    accepted: !!normalize(text) && manifest.decision.enabled && ![\'unknown\', \'out_of_scope\'].includes(ranked[0].label)\\n      && confidence >= manifest.decision.confidence && margin >= manifest.decision.margin,\\n    top: ranked.slice(0, 3), probabilities }\\n}\\n", "phone-test/worker.mjs": "import { loadModel, predict } from \'./runtime.mjs\'\\nlet model\\nself.onmessage = async ({ data }) => {\\n  try {\\n    if (data.type === \'load\') {\\n      const started = performance.now()\\n      model = await loadModel(data.manifest, data.buffer)\\n      self.postMessage({ type: \'loaded\', ms: performance.now() - started, id: model.manifest.model_id,\\n        enabled: model.manifest.decision.enabled, bytes: data.buffer.byteLength })\\n    } else if (data.type === \'predict\') {\\n      if (!model) throw Error(\'Load a model first\')\\n      const started = performance.now(), result = predict(model, data.text)\\n      self.postMessage({ type: \'prediction\', result, ms: performance.now() - started })\\n    } else if (data.type === \'benchmark\') {\\n      if (!model) throw Error(\'Load a model first\')\\n      const samples = []\\n      for (let i = 0; i < 10; i++) predict(model, data.text)\\n      for (let i = 0; i < 100; i++) { const start = performance.now(); predict(model, data.text); samples.push(performance.now() - start) }\\n      samples.sort((a, b) => a - b)\\n      self.postMessage({ type: \'benchmark\', p50: samples[49], p95: samples[94], max: samples[99] })\\n    }\\n  } catch (error) { self.postMessage({ type: \'error\', message: error.message }) }\\n}\\n"}')
for name, text in FILES.items():
    destination=ROOT/name
    destination.parent.mkdir(parents=True,exist_ok=True)
    destination.write_text(text,encoding='utf-8')
import numpy as np
print('NumPy',np.__version__,'; kit ready at',ROOT)


In [ ]:
from google.colab import files
UPLOADS=ROOT/'uploads'
UPLOADS.mkdir(exist_ok=True)
os.chdir(UPLOADS)
print('Upload all batch*.jsonl files. To replace a batch, upload its corrected file with the same name.')
uploaded=files.upload()
for name, content in uploaded.items():
    if not name.lower().endswith('.jsonl'): raise ValueError('Upload JSONL data only: '+name)
    (UPLOADS/Path(name).name).write_bytes(content)
os.chdir(ROOT)
DATA=sorted(str(p) for p in UPLOADS.glob('*.jsonl'))
if not DATA: raise ValueError('No JSONL files uploaded')
print('Dataset batches:',len(DATA))


In [ ]:
import subprocess
# Set True ONLY for a limited first-batch experiment; predictions cannot be accepted.
PROTOTYPE=False
FLAGS=['--prototype'] if PROTOTYPE else []
subprocess.run([sys.executable,str(ROOT/'validate_dataset.py'),*DATA,*FLAGS],cwd=ROOT,check=True)
# If this fails, repair the indicated rows/groups in Claude and rerun upload+validation.


In [ ]:
subprocess.run([sys.executable,str(ROOT/'train.py'),*DATA,*FLAGS,'--output',str(ROOT/'trained_model'),'--epochs','40','--seed','42'],cwd=ROOT,check=True)


In [ ]:
report=json.loads((ROOT/'trained_model/report.json').read_text())
print(json.dumps({key:report[key] for key in ['dataset','seconds','weight_bytes','decision']},indent=2))
print('Test:',json.dumps({k:v for k,v in report['test'].items() if k!='per_class'},indent=2))
print('Per-class results:')
for label, scores in report['test']['per_class'].items(): print(label,scores)
print('Bring model_bundle.zip back for an isolated HTTPS phone test. Production approval remains false.')


In [ ]:
files.download(str(ROOT/'model_bundle.zip'))
